# Equity Market Regime Analysis
### Identifying Bull, Bear, and Sideways Phases Using Unsupervised Machine Learning (K-Means, Hierarchical Clustering, and PCA)

Market regimes refer to distinct states of the financial markets (such as high volatility corrections, stable trends, or quiet consolidation periods) that dictate asset behavior. Traditional asset allocation models often assume constant return and volatility parameters. However, in reality, asset distributions are **non-stationary** and transition between different regimes.

In this notebook, we implement an unsupervised pipeline to identify these regimes:
1. **Feature Engineering**: Extract rolling trend, volatility, and downside risk indicators.
2. **Dimensionality Reduction (PCA)**: Compress correlated features to filter noise and project the data into a 2D space.
3. **K-Means Clustering**: Partition historical daily states into Bull, Bear, and Sideways groups.
4. **Hierarchical Clustering**: Build a dendrogram to visualize how market observations merge bottom-up based on feature distances.

### 1. Mathematical Background

#### Feature Normalization (Standardization)
Distance-based algorithms like K-Means and PCA are highly sensitive to scales. We standardize features to mean $\mu = 0$ and standard deviation $\sigma = 1$:
$$z = \frac{x - \mu}{\sigma}$$

#### Principal Component Analysis (PCA)
PCA projects our 3D feature space $[X_{ret}, X_{vol}, X_{dd}]$ onto orthogonal axes (principal components) that maximize variance. It decomposes the covariance matrix $\Sigma$:
$$\Sigma = \frac{1}{N-1} X^T X = V \Lambda V^T$$
where $V$ represents eigenvectors (loadings) and $\Lambda$ is the diagonal matrix of eigenvalues (variance explained).

#### K-Means Clustering
K-Means groups observations into $K$ clusters by minimizing the within-cluster sum of squares (inertia):
$$J = \sum_{j=1}^{K} \sum_{i \in S_j} \| z_i - \mu_j \|^2$$
where $\mu_j$ is the centroid of cluster $S_j$.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import scipy.cluster.hierarchy as sch

# Set plotting aesthetics
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    'figure.facecolor': '#090d16',
    'axes.facecolor': '#111827',
    'text.color': '#f8fafc',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': 'rgba(255, 255, 255, 0.05)',
    'font.family': 'sans-serif'
})
print("[+] Data science environment successfully loaded.")

In [ ]:
# Load Historical Market Prices (S&P 500 ETF as proxy)
ticker = "SPY"
df = yf.download(ticker, start="2020-01-01", end="2026-06-01")
df = df.reset_index()

# Flatten column headers if yfinance multi-index occurs
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df[['Date', 'Close']].copy()
df['Date'] = pd.to_datetime(df['Date'])
print(f"[*] Loaded {len(df)} days of historical data for {ticker}.")
df.head()

In [ ]:
# 2. Feature Engineering
# Lookback windows (typically 21 trading days represents ~1 calendar month)
lookback = 21

# Calculate Daily returns
df['Daily_Return'] = df['Close'].pct_change()

# Feature 1: Rolling returns (Momentum/Trend)
df['Rolling_Return'] = df['Close'].pct_change(periods=lookback)

# Feature 2: Rolling Volatility (annualized risk)
df['Rolling_Vol'] = df['Daily_Return'].rolling(window=lookback).std() * np.sqrt(252)

# Feature 3: Rolling Max Drawdown (Tail Risk)
rolling_max = df['Close'].rolling(window=lookback, min_periods=1).max()
df['Rolling_DD'] = (rolling_max - df['Close']) / rolling_max

# Clean NaNs resulting from rolling windows
df_clean = df.dropna().reset_index(drop=True)
print(f"[*] Cleaned dataset contains {len(df_clean)} records after features.")
df_clean.head()

In [ ]:
# 3. Standardize Features
feature_cols = ['Rolling_Return', 'Rolling_Vol', 'Rolling_DD']
X = df_clean[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("[*] Normalized feature sample:")
print(X_scaled[:5])

In [ ]:
# 4. PCA for Dimensionality Reduction
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"[*] Explained Variance Ratio by component: {pca.explained_variance_ratio_}")
print(f"[*] Cumulative Explained Variance: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

# Create loading DataFrame
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=feature_cols)
print("\n[*] PCA Loading Vectors (Eigenvectors):")
print(loadings)

In [ ]:
# 5. Unsupervised K-Means Clustering
k_clusters = 3
kmeans = KMeans(n_clusters=k_clusters, random_state=42, init='k-means++')
df_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# Re-index clusters logically based on economic attributes:
# Classify: Bear (0), Sideways (1), Bull (2)
cluster_profiles = []
for c in range(k_clusters):
    c_mask = (df_clean['Cluster'] == c)
    avg_r = df_clean.loc[c_mask, 'Rolling_Return'].mean()
    avg_v = df_clean.loc[c_mask, 'Rolling_Vol'].mean()
    cluster_profiles.append({'orig_id': c, 'avg_r': avg_r, 'avg_v': avg_v})

# Sort by average returns ascending (lowest return is Bear, highest is Bull)
sorted_profiles = sorted(cluster_profiles, key=lambda x: x['avg_r'])
label_map = {
    sorted_profiles[0]['orig_id']: 0,  # Bear
    sorted_profiles[1]['orig_id']: 1,  # Sideways
    sorted_profiles[2]['orig_id']: 2   # Bull
}

df_clean['Regime'] = df_clean['Cluster'].map(label_map)
print("[*] K-Means clustering completed & mapped to logical regimes.")

In [ ]:
# 6. Visualizing Market Price with Background Regimes
plt.figure(figsize=(15, 7.5))

dates = df_clean['Date'].values
prices = df_clean['Close'].values
regimes = df_clean['Regime'].values

colors = {0: '#f43f5e', 1: '#f59e0b', 2: '#10b981'}
names = {0: 'Bear Regime', 1: 'Sideways Regime', 2: 'Bull Regime'}

# Plot asset price line
plt.plot(dates, prices, color='#cbd5e1', linewidth=1.5, alpha=0.9, label=f"{ticker} Price")

# Shade areas according to active regime
for i in range(len(df_clean) - 1):
    r = regimes[i]
    plt.axvspan(dates[i], dates[i+1], color=colors[r], alpha=0.12, zorder=1)

# Plot color legend points
for r in [0, 1, 2]:
    mask = (regimes == r)
    plt.scatter(dates[mask], prices[mask], color=colors[r], s=3, label=names[r], zorder=2)

plt.title(f"{ticker} Close Price Overlayed with Unsupervised Market Regimes", fontsize=14, fontweight='semibold', pad=15)
plt.xlabel("Date", fontsize=11, labelpad=10)
plt.ylabel("Price ($)", fontsize=11, labelpad=10)
plt.legend(facecolor='#111827', edgecolor='none')
plt.show()

In [ ]:
# 7. 2D PCA Scatter Plot of Cluster Boundaries
plt.figure(figsize=(8, 7.5))
for r in [0, 1, 2]:
    mask = (regimes == r)
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], color=colors[r], label=names[r], alpha=0.6, s=15)

plt.title("Clustering Partition Projected in 2D PCA Space", fontsize=12, fontweight='semibold', pad=12)
plt.xlabel("Principal Component 1 (PC1)", fontsize=10, labelpad=8)
plt.ylabel("Principal Component 2 (PC2)", fontsize=10, labelpad=8)
plt.legend(facecolor='#111827', edgecolor='none')
plt.show()

In [ ]:
# 8. Hierarchical Clustering (Dendrogram)
# We perform hierarchical linkage on a subset (recent 150 days) for visual clarity
subset_days = 150
X_sub = X_scaled[-subset_days:]
dates_sub = df_clean['Date'].dt.strftime('%m-%d').values[-subset_days:]

linkage_matrix = sch.linkage(X_sub, method='average')

plt.figure(figsize=(15, 6))
sch.dendrogram(
    linkage_matrix, 
    labels=dates_sub,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=0.6 * max(linkage_matrix[:, 2]),
    above_threshold_color='#64748b'
)
plt.title("Hierarchical Agglomerative Dendrogram (Last 150 Days)", fontsize=12, fontweight='semibold', pad=12)
plt.xlabel("Trading Date (MM-DD)", fontsize=10, labelpad=8)
plt.ylabel("Euclidean Merge Distance", fontsize=10, labelpad=8)
plt.show()

In [ ]:
# 9. Summary Statistics of Regimes
summary_stats = []
for r in [0, 1, 2]:
    mask = (df_clean['Regime'] == r)
    sub_df = df_clean[mask]
    
    days = len(sub_df)
    pct = (days / len(df_clean)) * 100
    avg_return_21d = sub_df['Rolling_Return'].mean() * 100
    avg_vol_ann = sub_df['Rolling_Vol'].mean() * 100
    max_drawdown = sub_df['Rolling_DD'].max() * 100
    
    summary_stats.append({
        'Regime': names[r],
        'Observations (Days)': days,
        '% of Total': f"{pct:.1f}%",
        'Mean 21d Return': f"{avg_return_21d:+.2f}%",
        'Mean Ann Volatility': f"{avg_vol_ann:.2f}%",
        'Max Drawdown (21d)': f"{max_drawdown:.2f}%"
    })

pd.DataFrame(summary_stats)